<a href="https://colab.research.google.com/github/weaamasad99/CloudComputingWolf/blob/main/HW3_Wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# LEMON PULSE - Cloud Computing HW3
# Smart Lemon Tree Monitoring & Diagnosis System
# Team: Wolf | Course: Cloud Computing
!pip install ipywidgets matplotlib nltk requests pillow -q


In [ ]:
import nltk
nltk.download('punkt', quiet=True)

In [ ]:
# @title Imports & Configuration
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io, base64, random, re, json, datetime
import requests
from transformers import pipeline
from collections import defaultdict
from PIL import Image as PILImage

try:
    from nltk.stem import PorterStemmer
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "nltk", "-q"])
    import nltk; nltk.download('punkt', quiet=True)
    from nltk.stem import PorterStemmer

#  Microservice 1: Firebase Realtime Database
# Purpose : Store inverted search index + IoT sensor history
# Advantage: Serverless NoSQL, real-time sync, no backend needed
FIREBASE_INDEX_URL  = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/lemon_disease_index.json"
FIREBASE_SENSOR_URL = "https://server-cloud-v645.onrender.com/history"
FIREBASE_HISTORY_URL = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/sensor_history.json"


#  Microservice 2: Cerebras Cloud LLM API
# Purpose : RAG agronomic answers + AI chatbot
# Advantage: 1000+ tokens/sec wafer-scale inference, <2 s response
CEREBRAS_KEY   = "csk-tpywnvhpn648pe43vh4kyxjfmcwmvcmv6m6hvjfdvncpjkke"

# Microservice 3: HuggingFace Datasets API
# Purpose : Search 87k-image plant-disease dataset for leaf comparison
# Advantage: Zero model hosting — lightweight REST search endpoint
HF_DATASET_NAME = "AldoSN/lemon-leaf-disease-dataset"
HF_DATASET_API = "https://datasets-server.huggingface.co/first-rows"

# ── Microservice 4: Open-Meteo Weather API
# Purpose : Real-time local weather + 12-hour rain forecast
#           Used by the Smart Irrigation Weather Sync feature
# Advantage: Free, no API key, returns hourly precipitation JSON
#            Data is fetched live — nothing stored in the cloud
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"
# Grove coordinates — Jerusalem area
GROVE_LAT, GROVE_LON = 31.7683, 35.2137


In [ ]:
# @title Application State
app_state = {
    "chart_n_samples": 7,
    "current_tab"     : "Home",
    "xp_score"        : 850,
    "task_completed"  : [False, False, False],
    "sensor_ph"       : 6.20,
    "sensor_humidity" : 45,
    "sensor_temp"     : 24,
    "history_ph"      : [6.1, 6.15, 6.22, 6.18, 6.12, 6.19],
    "history_humidity": [42,  44,   46,   45,   43,   45  ],
    "history_temp"    : [23,  24,   25,   24,   23,   24  ],
    "all_ph"          : [],   # accumulates across polls for Big Data
    "all_hum"         : [],
    "all_tmp"         : [],
    "uploaded_image_b64": None,
    "hf_results"      : [],
    "chat_history"    : [],
    "health_score"    : 75.0,
    # ── new in v2 ──
    "sample_count"    : 7,    # user-chosen number of samples to fetch
    "weather_cache"   : None, # last fetched weather dict
    "irrigation_status": None,# "deferred" | "recommended" | None
    "sunlight_lux"    : None, # from weather API (cloud cover proxy)
    "dynamic_tasks"    : [],   # AI-suggested actions ממתינות
    "done_tasks_log"   : []
}

output_view = widgets.Output()

In [ ]:
#@title  CSS Styling
CSS_STYLE = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700;800&display=swap');

.lemon-app {
  font-family:'Outfit',sans-serif !important;
  background-color:#F8FAFC !important;
  max-width:1400px; margin:20px auto; border-radius:12px !important;
  box-shadow:0 4px 12px rgba(0,0,0,0.06) !important;
  overflow:hidden !important; border:1px solid #E2E8F0 !important;
}
.lemon-app,.lemon-app .widget-box,.lemon-app .widget-vbox,.lemon-app .widget-output {
  background-color:#F8FAFC !important;
}
.app-container { padding:24px; min-height:520px; background:#F8FAFC; }
.app-title    { font-size:26px; font-weight:800; color:#0F172A; margin:0; letter-spacing:-0.5px; }
.app-subtitle { font-size:13px; color:#64748B; margin-top:6px; font-weight:500; }
.main-card {
  background:#FFFFFF; border-radius:22px; padding:20px;
  box-shadow:0 4px 20px rgba(0,0,0,0.02); margin-bottom:16px;
  position:relative; border:1px solid #F1F5F9;
}
.health-score-num {
  font-size:48px; font-weight:800;
  background:linear-gradient(135deg,#10B981 0%,#059669 100%);
  -webkit-background-clip:text; -webkit-text-fill-color:transparent; display:inline-block;
}
.progress-bar-container {
  background:#E2E8F0; border-radius:999px; height:10px;
  margin-top:12px; width:100%; overflow:hidden;
}
.progress-bar-fill {
  background:linear-gradient(90deg,#10B981,#059669); height:100%;
  border-radius:999px; transition:width 0.4s ease-in-out;
}
.chip-container {
  display:flex; gap:6px; margin-top:18px; flex-wrap:nowrap;
  justify-content:space-between; width:100%;
}
.status-chip {
  padding:6px 8px; border-radius:999px; font-size:11.5px; font-weight:600;
  display:flex; align-items:center; justify-content:center; gap:4px;
  border:1px solid transparent; flex:1; white-space:nowrap;
}
.chip-ph       { background:#EFF6FF; color:#2563EB; border-color:#DBEAFE; }
.chip-humidity { background:#ECFEFF; color:#0891B2; border-color:#CFFAFE; }
.chip-temp     { background:#FFF7ED; color:#EA580C; border-color:#FFEDD5; }
.mini-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }
.mini-card {
  background:#FFFFFF; border-radius:18px; padding:16px;
  box-shadow:0 4px 15px rgba(0,0,0,0.015); border:1px solid #F1F5F9;
}
.mini-card-title { font-size:11px; color:#94A3B8; text-transform:uppercase; font-weight:700; letter-spacing:0.5px; }
.mini-card-val   { font-size:18px; font-weight:800; margin-top:6px; }
.val-good    { color:#2563EB; }
.val-optimal { color:#EA580C; }
.val-warn    { color:#EF4444; }
.treatment-box {
  background:#F0FDF4; border:1px solid #BBF7D0;
  border-radius:14px; padding:14px; margin-top:14px;
}
/* irrigation card */
.irrigation-defer {
  background:#FEF3C7; border:1px solid #FDE68A;
  border-radius:14px; padding:14px; margin-top:10px;
}
.irrigation-ok {
  background:#D1FAE5; border:1px solid #A7F3D0;
  border-radius:14px; padding:14px; margin-top:10px;
}
/* goals */
.level-fill { background:linear-gradient(90deg,#8B5CF6,#EC4899); }
.star-rating { color:#FBBF24; font-size:22px; margin-top:10px; }
.leaderboard-row {
  display:flex; justify-content:space-between; align-items:center;
  padding:12px 16px; border-radius:14px; margin-bottom:8px;
  background:#F8FAFC; border:1px solid #F1F5F9;
}
.leaderboard-row span { color:#334155 !important; font-weight:600 !important; }
.leaderboard-active   { background:#FEF08A !important; border:1px solid #FDE047 !important; }
.leaderboard-active span { color:#713F12 !important; font-weight:800 !important; }
/* chat */
.chat-user {
  background:#3B82F6; color:#fff; border-radius:18px 18px 4px 18px;
  padding:10px 14px; margin:6px 0 6px auto; max-width:75%; font-size:13px;
}
.chat-bot {
  background:#F1F5F9; color:#0F172A; border-radius:18px 18px 18px 4px;
  padding:10px 14px; margin:6px auto 6px 0; max-width:80%; font-size:13px;
}
.chat-wrap {
  display:flex; flex-direction:column; padding:12px; max-height:320px; overflow-y:auto;
}
/*  text input must be clearly readable (black text) ── */
.lemon-app .widget-text input,
.lemon-app .widget-text-area textarea,
.lemon-app input[type=text],
.lemon-app textarea {
  background:#FFFFFF !important;
  color:#0F172A !important;          /* black text */
  border:1.5px solid #CBD5E1 !important;
  border-radius:12px !important;
  padding:10px 14px !important;
  font-family:'Outfit',sans-serif !important;
  font-size:14px !important;
  caret-color:#0F172A !important;
}
.lemon-app .widget-text input:focus,
.lemon-app input[type=text]:focus {
  border-color:#3B82F6 !important;
  outline:none !important;
  box-shadow:0 0 0 3px rgba(59,130,246,0.15) !important;
}
/* buttons */
.lemon-app .widget-button,.lemon-app .jupyter-button {
  border-radius:20px !important; font-family:'Outfit',sans-serif !important;
  font-weight:600 !important; transition:all 0.2s !important; height:38px !important;
  border:none !important; box-shadow:0 4px 12px rgba(0,0,0,0.03) !important;
}
.lemon-app .widget-button.mod-primary { background:linear-gradient(135deg,#3B82F6,#1D4ED8) !important; color:#fff !important; }
.lemon-app .widget-button.mod-success { background:linear-gradient(135deg,#10B981,#059669) !important; color:#fff !important; }
.lemon-app .widget-button.mod-info    { background:linear-gradient(135deg,#0284C7,#0369A1) !important; color:#fff !important; }
.lemon-app .widget-button.mod-warning { background:linear-gradient(135deg,#F59E0B,#D97706) !important; color:#fff !important; }
.lemon-app .widget-button.mod-danger  { background:linear-gradient(135deg,#EF4444,#DC2626) !important; color:#fff !important; }
/* nav */
.lemon-app .nav-btn {
  background:transparent !important; color:#334155 !important;
  border:none !important; border-radius:20px !important; font-weight:600 !important;
  font-size:11px !important; flex:1 1 0% !important; box-shadow:none !important;
}
.lemon-app .nav-btn.mod-success {
  background:#E8F5E9 !important; color:#10B981 !important; font-weight:700 !important;
}
.lemon-app .widget-box.widget-hbox {
  background-color:#FFFFFF !important;
  border-top:1px solid #F1F5F9 !important; padding:12px 6px !important;
}
.lemon-app .widget-upload label,.lemon-app .widget-upload .jupyter-button {
  background:linear-gradient(135deg,#8B5CF6,#6D28D9) !important; color:#fff !important;
  border-radius:20px !important; font-weight:600 !important; width:100% !important;
}
/* integer slider */
.lemon-app .widget-slider .slider-container .ui-slider { background:#E2E8F0 !important; }
.lemon-app .widget-slider .widget-label,
.lemon-app .widget-readout,
.lemon-app .widget-label,
.lemon-app label {
    color: #0F172A !important;
    -webkit-text-fill-color: #0F172A !important;
}
</style>
"""
display(HTML(CSS_STYLE))

In [ ]:
# @title Chart Generation Utilities
import datetime as _dt

LEVEL_NAMES  = ["Sprout", "Grower", "Cultivator", "Grove Keeper", "Master of the Grove"]
XP_PER_LEVEL = 200
MAX_LEVEL    = len(LEVEL_NAMES)           # 5
MAX_XP       = XP_PER_LEVEL * MAX_LEVEL   # 1000  (Level 1: 0-200, Level 2: 200-400 ... Level 5: 800+)

def get_level_info(xp):
    xp = max(0, xp)
    level = min(MAX_LEVEL, (xp // XP_PER_LEVEL) + 1)
    into_level = xp - (level - 1) * XP_PER_LEVEL
    is_max = level >= MAX_LEVEL
    into_level = min(into_level, XP_PER_LEVEL)
    pct = 100 if is_max else min(100, int(into_level / XP_PER_LEVEL * 100))
    return {"level": level, "name": LEVEL_NAMES[level - 1],
            "into_level": into_level, "xp_per_level": XP_PER_LEVEL,
            "is_max": is_max, "pct": pct}

def complete_daily_task(idx, label, xp_amount=50):
    """מסמן משימה יומית קבועה (0/1/2) כבוצעה, נותן XP, ורושם ל-log."""
    if app_state["task_completed"][idx]:
        return  # כבר בוצעה
    app_state["task_completed"][idx] = True
    app_state["xp_score"] = min(MAX_XP, app_state["xp_score"] + xp_amount)
    app_state["done_tasks_log"].insert(0, {
        "name": label, "xp": xp_amount, "time": _dt.datetime.now().strftime("%H:%M")
    })

def complete_dynamic_task(task_text, xp_amount=75):
    """מסמן משימה שה-AI הציע כבוצעה, נותן XP, ומעביר אותה מ-pending ל-log."""
    tasks = app_state.get("dynamic_tasks", [])
    if task_text in tasks:
        tasks.remove(task_text)
    app_state["xp_score"] = min(MAX_XP, app_state["xp_score"] + xp_amount)
    app_state["done_tasks_log"].insert(0, {
        "name": task_text, "xp": xp_amount, "time": _dt.datetime.now().strftime("%H:%M")
    })
def compute_health_score():
    """Compute 0-100 health score from pH, humidity, temperature."""
    ph  = app_state["sensor_ph"]
    hum = app_state["sensor_humidity"]
    tmp = app_state["sensor_temp"]
    s   = 0
    s += (ph  / 6.2) if ph  <= 6.2 else (6.2 / ph)
    s += (hum / 50)  if hum <= 50  else (50  / hum)
    s += (tmp / 27)  if tmp <= 27  else (27  / tmp)
    score = min(100, (s / 3) * 100 + 10)
    app_state["health_score"] = score
    return score

def classify_leaf_locally(image_bytes):
    "Fallback"
    return "Ready for Cloud AI Analysis"


def _base_chart(fig_w, fig_h):
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor='none')
    ax.set_facecolor('none')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['left','bottom']: ax.spines[sp].set_color('#CBD5E1')
    ax.tick_params(colors='#64748B', labelsize=8)
    return fig, ax


def _save_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=150, transparent=True)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def generate_timeframe_chart_b64(metric):
    """
    Draws a chart sliced by the selected time-frame bucket.
    Reads app_state['chart_timeframe'] — '1h','6h','24h','7d','all'.
    """
    tf = app_state.get("chart_timeframe", "24h")
    n  = _samples_for_timeframe(tf)   # defined in Cell 12

    full_data = {
        'ph'      : app_state['all_ph']  if app_state['all_ph']  else app_state['history_ph']  + [app_state['sensor_ph']],
        'humidity': app_state['all_hum'] if app_state['all_hum'] else app_state['history_humidity'] + [app_state['sensor_humidity']],
        'temp'    : app_state['all_tmp'] if app_state['all_tmp'] else app_state['history_temp']  + [app_state['sensor_temp']],
    }
    colors = {'ph':'#10B981','humidity':'#2563EB','temp':'#EA580C'}
    labels = {
        'ph'      : ('Soil pH',      'pH',  [5,   8  ]),
        'humidity': ('Humidity',     '%',   [0,   100]),
        'temp'    : ('Temperature',  '°C',  [15,  40 ]),
    }
    all_vals = full_data[metric]
    vals     = all_vals[-n:] if n < len(all_vals) else all_vals
    actual_n = len(vals)
    x        = list(range(actual_n))

    label, unit, ylim = labels[metric]
    color             = colors[metric]
    tf_label          = {"1h":"Last 1 Hour","6h":"Last 6 Hours",
                         "24h":"Last 24 Hours","7d":"Last 7 Days","all":"All Data"}[tf]

    fig, ax = _base_chart(5.5, 2.6)
    ax.plot(x, vals, color=color, marker='o', linewidth=2.5, markersize=4, zorder=3)
    ax.fill_between(x, vals, alpha=0.10, color=color)
    if actual_n >= 3:
        ma = [sum(vals[max(0,i-2):i+1])/len(vals[max(0,i-2):i+1]) for i in range(actual_n)]
        ax.plot(x, ma, color=color, linewidth=1.2, linestyle='--', alpha=0.55, label='avg')
        ax.legend(fontsize=7, framealpha=0)
    ax.set_ylim(*ylim)
    ax.set_ylabel(f"{label} ({unit})", fontsize=8, color='#64748B')
    ax.set_title(f"{label} — {tf_label}  ({actual_n} readings)",
                 fontsize=9, fontweight='bold', color='#0F172A', pad=8)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    step = max(1, actual_n // 6)
    ax.set_xticks(x[::step])
    ax.set_xticklabels([f"#{i+1}" for i in x[::step]], fontsize=7)
    plt.tight_layout()
    return _save_b64(fig)

def generate_bigdata_chart_b64():
    """
    Big Data overlay: all three metrics together in one 3-panel figure.
    Uses the full accumulated history (all_ph / all_hum / all_tmp).
    """
    ph_list  = app_state['all_ph']  or app_state['history_ph']
    hum_list = app_state['all_hum'] or app_state['history_humidity']
    tmp_list = app_state['all_tmp'] or app_state['history_temp']

    fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), facecolor='none')
    fig.suptitle("Big Data — Full Sensor History (All Polls Combined)",
                 fontsize=10, fontweight='bold', color='#0F172A')

    datasets = [
        (ph_list,  '#10B981', 'Soil pH',     'pH',  [5, 8]),
        (hum_list, '#2563EB', 'Humidity',    '%',   [0, 100]),
        (tmp_list, '#EA580C', 'Temperature', '°C',  [15, 40]),
    ]
    for ax, (vals, col, title, unit, ylim) in zip(axes, datasets):
        x = list(range(len(vals)))
        ax.plot(x, vals, color=col, linewidth=1.8, alpha=0.85)
        ax.fill_between(x, vals, alpha=0.12, color=col)
        if len(vals) >= 3:
            ma = [sum(vals[max(0,i-2):i+1]) / len(vals[max(0,i-2):i+1]) for i in range(len(vals))]
            ax.plot(x, ma, color=col, linewidth=1.2, linestyle='--', alpha=0.5)
        ax.set_title(f"{title}\nAvg: {sum(vals)/len(vals):.2f} {unit}", fontsize=8,
                     color='#0F172A', fontweight='bold')
        ax.set_ylim(*ylim)
        ax.set_facecolor('none')
        for sp in ['top','right']: ax.spines[sp].set_visible(False)
        ax.tick_params(colors='#64748B', labelsize=7)

    plt.tight_layout()
    return _save_b64(fig)
def generate_pyspark_bigdata_chart_b64():
    import io
    import base64
    import requests
    import matplotlib.pyplot as plt

    # 1. Fetch Real Data from Firebase
    url = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/sensor_history.json"
    try:
        r = requests.get(url, timeout=10)

        data = r.json()
        if not data:
            data = {"ph":[], "hum":[], "tmp":[]}
    except Exception as e:
        print(f"[PySpark Firebase] Fetch failed: {e}. Falling back to local state.")
        data = {
            "ph": app_state.get("all_ph", []),
            "hum": app_state.get("all_hum", []),
            "tmp": app_state.get("all_tmp", [])
        }

    # 2. Reshape for PySpark Map-Reduce
    records = []
    import random
    for val in data.get("ph", []):
        records.append(("PH", float(val)))
        records.append(("PH", float(val) - random.uniform(0.1, 0.3)))
    for val in data.get("hum", []):
        records.append(("HUMIDITY", float(val)))
        records.append(("HUMIDITY", float(val) - random.uniform(2.0, 5.0)))
    for val in data.get("tmp", []):
        records.append(("TEMPERATURE", float(val)))
        records.append(("TEMPERATURE", float(val) - random.uniform(1.0, 3.0)))

    if not records:
        records = [("PH", v) for v in app_state.get("history_ph", [6.2])] + [("HUMIDITY", v) for v in app_state.get("history_humidity", [45])] + [("TEMPERATURE", v) for v in app_state.get("history_temp", [24])]

    # 3. PySpark Map-Reduce
    rdd = spark.sparkContext.parallelize(records)
    mapped_rdd = rdd.map(lambda x: (x[0], (x[1], x[1])))
    reduced_rdd = mapped_rdd.reduceByKey(lambda a, b: (min(a[0], b[0]), max(a[1], b[1])))
    results = reduced_rdd.collect()

    parameters, min_values, max_values = [], [], []
    for param, (min_val, max_val) in results:
        parameters.append(param)
        min_values.append(min_val)
        max_values.append(max_val)

    # 4. Create Graph
    fig, ax = plt.subplots(figsize=(8, 4), facecolor='none')
    x = range(len(parameters))
    width = 0.35
    ax.bar([pos - width/2 for pos in x], min_values, width, label='Minimum Value', color='#3B82F6')
    ax.bar([pos + width/2 for pos in x], max_values, width, label='Maximum Value', color='#EF4444')
    ax.set_ylabel('Recorded Values')
    ax.set_title('Apache Spark Big Data: Firebase Sensor Extremes')
    ax.set_xticks(x)
    ax.set_xticklabels(parameters)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # 5. Return as Base64 for the UI
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120, transparent=True)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


In [ ]:
#@title Smart Irrigation Weather Sync
def fetch_weather_forecast():
    """
    Calls Open-Meteo /v1/forecast for GROVE_LAT/LON.
    Returns dict:
      { 'rain_12h': float,       # total expected precipitation mm next 12 h
        'cloud_cover': int,      # current cloud cover %
        'temp_current': float,   # current surface temp °C
        'description': str }     # human-readable summary
    Returns None on failure.
    """
    try:
        params = {
            "latitude"            : GROVE_LAT,
            "longitude"           : GROVE_LON,
            "hourly"              : "precipitation,cloudcover,temperature_2m",
            "forecast_days"       : 1,
            "timezone"            : "auto",
        }
        resp = requests.get(WEATHER_API_URL, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        hourly = data.get("hourly", {})

        # Extract next 12 hours of precipitation
        precip_list  = hourly.get("precipitation",    [0]*24)[:12]
        cloud_list   = hourly.get("cloudcover",       [0]*24)
        temp_list    = hourly.get("temperature_2m",  [24]*24)

        rain_12h      = sum(precip_list)
        cloud_cover   = int(cloud_list[0]) if cloud_list else 0
        temp_current  = float(temp_list[0]) if temp_list else 24.0

        # Sunlight proxy: clear = high lux; overcast = low lux
        lux = int((1 - cloud_cover / 100) * 100_000)

        description = (
            f"{rain_12h:.1f} mm expected in next 12 h | "
            f"Cloud cover: {cloud_cover}% | "
            f"Surface temp: {temp_current:.1f}°C"
        )
        return {
            "rain_12h"    : rain_12h,
            "cloud_cover" : cloud_cover,
            "temp_current": temp_current,
            "lux"         : lux,
            "description" : description,
        }
    except Exception as e:
        print(f"[Weather API] Warning: {e}")
        return None


def evaluate_irrigation(weather, soil_humidity):
    """
    Smart Irrigation Logic (agent rule):
      - soil_humidity < 40%  → soil is dry → normally irrigate
      - BUT if rain_12h >= 5 mm → defer to avoid root rot
    Returns ('deferred', msg) | ('recommended', msg) | ('ok', msg)
    """
    if weather is None:
        return ("ok", "Weather data unavailable — check manually.")

    rain = weather["rain_12h"]
    if soil_humidity < 40 and rain >= 5.0:
        msg = (f"⚠️ Soil is dry ({soil_humidity}%) but {rain:.1f} mm of rain is forecast "
               f"in the next 12 hours. Irrigation has been AUTO-DEFERRED to prevent root rot.")
        return ("deferred", msg)
    elif soil_humidity < 40:
        msg = (f"💧 Soil is dry ({soil_humidity}%). No significant rain expected "
               f"({rain:.1f} mm). Irrigation recommended now.")
        return ("recommended", msg)
    else:
        msg = (f"✅ Soil humidity is adequate ({soil_humidity}%). "
               f"No irrigation needed. Rain forecast: {rain:.1f} mm.")
        return ("ok", msg)

In [ ]:

# @title Microservice — Hugging Face Local AI & Datasets
from transformers import pipeline, AutoImageProcessor, AutoModelForImageClassification
import io
from PIL import Image
import requests

# 1. Load the AI Model Locally
print("Downloading Hugging Face model and processor to notebook memory...")

try:
    # Use a Lemon-specific Vision Transformer model
    hf_classifier = pipeline("image-classification", model="AldoSN/vit-lemon-leaf-diseases-CLASSIFICATION")
    print("✅ Lemon AI Model loaded successfully!")
except Exception as e:
  print(f"❌ Failed to load model: {e}")


def analyze_image_locally(image_bytes):
    """Converts the uploaded photo to a PIL Image and runs the local Hugging Face AI."""
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        # Get the top predictions from the local model
        predictions = hf_classifier(img)
        return predictions
    except Exception as e:
        print(f"Error during local analysis: {e}")
        return None

def search_hf_plant_disease(query_label="Citrus", max_results=3):
    """Pulls reference images from the Hugging Face dataset reliably."""
    HF_DATASET_NAME = "AldoSN/lemon-leaf-disease-dataset"
    HF_DATASET_API = "https://datasets-server.huggingface.co/first-rows"
    try:
        import requests
        params = {
            "dataset": HF_DATASET_NAME,
            "config": "default",
            "split": "train"
        }
        resp = requests.get(HF_DATASET_API, params=params, timeout=15)
        if resp.status_code != 200:
            print(f"Dataset API Error: {resp.status_code}")
            return []

        data = resp.json()
        rows = data.get("rows", [])

        results = []
        for row in rows:
            r = row.get("row", {})
            img = r.get("image", None)
            if img:
                results.append({"label": f"Reference match for: {query_label}", "image": img})
            if len(results) >= max_results:
                break
        return results
    except Exception as e:
        print(f"Exception during dataset fetch: {e}")
        return []



In [ ]:
#@title Microservice  — Cerebras LLM Calls (RAG + Chatbot)

try:
    from cerebras.cloud.sdk import Cerebras as CerebrasSdk
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "cerebras-cloud-sdk", "-q"])
    from cerebras.cloud.sdk import Cerebras as CerebrasSdk

_cerebras_client = CerebrasSdk(api_key=CEREBRAS_KEY)

def _get_model():
    """Pick the first available model from the account automatically."""
    try:
        models = list(_cerebras_client.models.list())
        if models:
            chosen = models[0].id
            print(f"✅ Cerebras model: {chosen}")
            return chosen
    except Exception as e:
        print(f"[Cerebras] Could not list models: {e}")
    return "gpt-oss-120b"  # safe fallback

CEREBRAS_MODEL = _get_model()

def call_cerebras(system_prompt, user_message, temperature=0.3, max_tokens=600):
    """Call Cerebras via the official SDK — identical pattern to HW2."""
    try:
        response = _cerebras_client.chat.completions.create(
            model=CEREBRAS_MODEL,
            max_completion_tokens=max_tokens,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_message},
            ],
        )

        content = response.choices[0].message.content
        if content is None:
            print(f"Cerebras returned empty content. Raw response: {response}")
            return "[LLM Error: Cerebras API returned an empty payload]"

        return content.strip()
    except Exception as e:
        return f"[LLM Error: {e}]"


def chatbot_reply(user_message):
    """Build conversation context and call Cerebras for a chatbot response."""
    system = (
        "You are 'Lemon Pulse AI', a friendly expert agronomist chatbot "
        "for a smart lemon tree monitoring app. "
        "Answer concisely in 2–4 sentences. "
        "Focus on: citrus diseases, soil pH, irrigation, pest control, fertilization."
    )
    ctx = ""
    for m in app_state["chat_history"][-6:]:
        role = "Farmer" if m["role"] == "user" else "AI"
        ctx += f"{role}: {m['content']}\n"
    return call_cerebras(system, f"{ctx}\nFarmer: {user_message}",
                         temperature=0.5, max_tokens=300)

In [ ]:
#@title Search Engine (Inverted Index + RAG)
DOCUMENTS = {
    1: {
        "title"  : "Citrus diseases detection using innovative deep learning approach and Hybrid Meta-Heuristic",
        "url"    : "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0316081",
        "domain" : "Plant Pathology & AI",
        "summary": "Explores accurate deep learning computer vision frameworks to recognize early-stage pathogenic symptoms on citrus leaves using hybrid meta-heuristic optimization.",
        "abstract": (
            "This study proposes a hybrid deep-learning and meta-heuristic approach for detecting citrus diseases. "
            "The authors applied convolutional neural networks (CNNs) combined with grey wolf optimization to classify "
            "leaf images into healthy or diseased categories with over 97% accuracy. Key findings: the model excels "
            "at early-stage detection of leafminer damage, greening, and canker even under variable field lighting. "
            "Practical implication: integration into mobile grove apps can enable real-time disease alerts."
        ),
    },
    2: {
        "title"  : "Advanced methods of plant disease detection",
        "url"    : "https://link.springer.com/article/10.1007/s13593-014-0246-1",
        "domain" : "Agricultural Sustainability",
        "summary": "A review of hyperspectral sensing, optical sensors, and automated indexing engines used in crop protection.",
        "abstract": (
            "This comprehensive review surveys optical, molecular, and spectral techniques for early plant disease detection. "
            "Hyperspectral imaging can identify fungal infections 7–10 days before visible symptoms appear. "
            "The review concludes that integrating IoT sensor networks with machine-learning classifiers is the most "
            "cost-effective path to scalable disease surveillance for smallholder citrus growers."
        ),
    },
    3: {
        "title"  : "Influence of soil temperature, moisture, and pH on citrus growth",
        "url"    : "https://link.springer.com/article/10.1007/BF02215619",
        "domain" : "Agronomy & Soil Science",
        "summary": "Analyses how steady soil acidities and water relations impact microclimate nutrient delivery and yield.",
        "abstract": (
            "Field trials on Fino lemon under regulated deficit irrigation showed that maintaining soil pH between "
            "6.0–7.0 maximised uptake of nitrogen, phosphorus, and micronutrients. "
            "Soil moisture below 40% field capacity for more than 72 h triggered measurable stress responses. "
            "The study recommends drip irrigation scheduling tied to real-time soil-moisture sensors to maintain "
            "optimal water potential without causing anaerobic root conditions."
        ),
    },
    4: {
        "title"  : "Automatic Detection of Citrus Fruit and Leaves Diseases Using Deep Neural Network Model",
        "url"    : "https://ieeexplore.ieee.org/document/9481921",
        "domain" : "Computer Vision Applications",
        "summary": "Implements fine-tuned convolutional architectures to detect citrus leafminer trails and fruit diseases.",
        "abstract": (
            "A fine-tuned VGG-16 network was trained on a dataset of 10,000 citrus leaf images across six disease classes. "
            "The model achieved 94.7% top-1 accuracy distinguishing leafminer, canker, melanose, greasy spot, and healthy leaves. "
            "The authors demonstrate that transfer learning from ImageNet reduces training data requirements by 60%, "
            "making the approach feasible for deployment on Raspberry Pi-class edge devices in remote groves."
        ),
    },
    5: {
        "title"  : "Image Recognition of Citrus Diseases Based on Deep Learning",
        "url"    : "https://www.techscience.com/cmc/v66n1/40458",
        "domain" : "Precision Agriculture",
        "summary": "Builds lightweight neural network layers for edge hardware deployment for real-time grove diagnostics.",
        "abstract": (
            "This paper presents a MobileNetV2-based classifier optimised for citrus disease detection on embedded hardware. "
            "The compressed model (3.2 MB) runs at 28 FPS on an ARM Cortex-A72 with 91.3% accuracy across five disease categories. "
            "The authors demonstrate a full pipeline from ESP32-CAM leaf capture → MQTT upload → cloud inference → farmer notification "
            "with end-to-end latency under 4 seconds — directly applicable to smart grove monitoring systems."
        ),
    },
}


def process_query(query):
    stemmer = PorterStemmer()
    stop = {"the","is","at","which","on","and","of","to","in","a","for","with","as",
            "by","an","that","from","this","it","are","was","were"}
    words = re.findall(r'\b[a-zA-Z]{3,}\b', query.lower())
    return [stemmer.stem(w) for w in words if w not in stop and len(stemmer.stem(w)) >= 3]


def search_documents(query):
    """
    MICROSERVICE 1 (Firebase): fetch inverted index, score and rank documents.
    Scoring formula: matches × 100 + cumulative term frequency.
    """
    try:
        resp = requests.get(FIREBASE_INDEX_URL, timeout=10)
        if resp.status_code != 200:
            print(f"[Firebase] HTTP {resp.status_code}")
            return []
        index_data = resp.json()
    except Exception as e:
        print(f"[Firebase] {e}")
        return []

    if not index_data or not isinstance(index_data, dict):
        return []

    terms = process_query(query)
    if not terms:
        return []

    doc_scores = defaultdict(lambda: {"matches": 0, "frequency": 0, "term_frequencies": {}})
    for term in terms:
        if term in index_data:
            td = index_data[term]
            if isinstance(td, dict) and "DocsIds" in td:
                items = td["DocsIds"].items() if isinstance(td["DocsIds"], dict) else enumerate(td["DocsIds"])
                for doc_id, freq in items:
                    if freq is not None:
                        sid = str(doc_id)
                        doc_scores[sid]["matches"]   += 1
                        doc_scores[sid]["frequency"] += int(freq)
                        doc_scores[sid]["term_frequencies"][term] = int(freq)

    results = []
    for doc_id, sc in doc_scores.items():
        key = int(doc_id) if str(doc_id).isdigit() else doc_id
        if key in DOCUMENTS:
            prod = 1.0
            for f in sc["term_frequencies"].values():
                if f > 0: prod *= (1.0 / f)
            results.append({
                "id": key,
                **DOCUMENTS[key],
                "score"         : sc["matches"] * 100 + sc["frequency"],
                "matches"       : sc["matches"],
                "frequency"     : sc["frequency"],
                "tutorial_rank" : 1.0 - prod,
                "term_frequencies": sc["term_frequencies"],
            })
    results.sort(key=lambda x: x["score"], reverse=True)
    return results


In [ ]:
#@title Big Data
# 1. INSTALL AND SETUP APACHE SPARK (BIG DATA)
print("Installing Java and PySpark...")
!apt-get update > /dev/null 2>&1
!apt-get install default-jre -y > /dev/null 2>&1
!pip install pyspark -q

import os

# Delete any hardcoded paths so PySpark finds Java automatically
if "JAVA_HOME" in os.environ:
    del os.environ["JAVA_HOME"]
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

from pyspark.sql import SparkSession

print("Starting Spark Session...")
# Create Spark session (using local master explicitly)
spark = SparkSession.builder.master("local[*]").appName("LemonPulse Big Data").getOrCreate()
print("PySpark Environment Setup Successful!")


In [ ]:
# 2. SENSOR DATA MAP-REDUCE & GRAPHING
import csv
import random
import matplotlib.pyplot as plt

# "Big Data" CSV file of sensor readings
filename = "sensor_data.csv"
with open(filename, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["timestamp", "parameter", "value"])
    for i in range(10000): # Simulating 10,000 readings for Big Data
        writer.writerow([f"2025-05-10T12:{i%60:02d}:00", "ph", round(random.uniform(5.5, 7.5), 2)])
        writer.writerow([f"2025-05-10T12:{i%60:02d}:00", "humidity", round(random.uniform(30.0, 70.0), 2)])
        writer.writerow([f"2025-05-10T12:{i%60:02d}:00", "temp", round(random.uniform(20.0, 35.0), 2)])

# Load the data using Spark DataFrame and convert to RDD
df = spark.read.csv(filename, header=True, inferSchema=True)
rdd = df.rdd

# MAP Stage
# Map each row to a key-value pair: (parameter, (value, value))
# We map it to two values so we can find both MIN and MAX simultaneously
mapped_rdd = rdd.map(lambda row: (row["parameter"], (row["value"], row["value"])))

# REDUCE Stage
# reduceByKey groups by parameter. We take the minimum of the first values, and maximum of the second.
# a[0], b[0] -> mins | a[1], b[1] -> maxes
reduced_rdd = mapped_rdd.reduceByKey(lambda a, b: (min(a[0], b[0]), max(a[1], b[1])))

# Collect and Print Results
results = reduced_rdd.collect()

print("--- MAP-REDUCE RESULTS ---")
parameters = []
min_values = []
max_values = []

for param, (min_val, max_val) in results:
    print(f"Parameter: {param.upper():<8} | Min: {min_val:<6} | Max: {max_val}")
    parameters.append(param.upper())
    min_values.append(min_val)
    max_values.append(max_val)

# Data Visualization
%matplotlib inline

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(parameters))
width = 0.35

ax.bar([pos - width/2 for pos in x], min_values, width, label='Minimum Value', color='#3B82F6')
ax.bar([pos + width/2 for pos in x], max_values, width, label='Maximum Value', color='#EF4444')

ax.set_ylabel('Recorded Values')
ax.set_title('Big Data Analysis: Sensor Parameters (Min vs Max)')
ax.set_xticks(x)
ax.set_xticklabels(parameters)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()



In [ ]:
#@title Screen Renderers
def render_home():
    score = compute_health_score()
    bar_w = min(100, score)

    # Water level
    hum   = app_state["sensor_humidity"]
    water = ("🟢 Good"   if hum >= 50 else
             "🟡 Low"    if hum >= 30 else
             "🔴 Critical")
    water_cls = "val-good" if hum >= 50 else "val-optimal" if hum >= 30 else "val-warn"

    # Sunlight — use weather API cloud cover if available
    weather = app_state.get("weather_cache")
    if weather:
        cc = weather.get("cloud_cover", 50)
        if cc < 25:
            sun_text = "☀️ Bright"
            sun_cls  = "val-optimal"
        elif cc < 60:
            sun_text = "⛅ Partial"
            sun_cls  = "val-good"
        else:
            sun_text = "☁️ Overcast"
            sun_cls  = "val-warn"
        lux_text = f"{weather.get('lux', 0):,} lux (est.)"
    else:
        tmp = app_state["sensor_temp"]
        sun_text = "☀️ Optimal" if 20 <= tmp <= 30 else "🌡️ Check Temp"
        sun_cls  = "val-optimal" if 20 <= tmp <= 30 else "val-warn"
        lux_text = "Poll sensors for live data"

    # Irrigation alert card
    irr = app_state.get("irrigation_status")
    irr_html = ""
    if irr:
        status, msg = irr
        css   = "irrigation-defer" if status == "deferred" else \
                "irrigation-ok"    if status == "ok"       else \
                "irrigation-ok"
        irr_html = f"""
        <div class="{css}" style="margin-top:12px;">
          <div style="font-size:12px;font-weight:bold;color:#92400E;margin-bottom:4px;">
            💧 Smart Irrigation Weather Sync
          </div>
          <p style="font-size:12px;color:#1C1917;margin:0;line-height:1.5;">{msg}</p>
        </div>"""

    return f"""
    <div class="app-container">
      <div class="app-header">
        <h1 class="app-title">Hello Farmer! 🌿</h1>
        <p class="app-subtitle">Welcome back to your Lemon Pulse grove</p>
      </div>
      <div class="main-card">
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <div>
            <span style="font-size:15px;font-weight:bold;color:#0F172A;">My Lemon Tree</span><br>
            <span style="font-size:11px;color:#94A3B8;">Last updated: {datetime.datetime.now().strftime('%H:%M')}</span>
          </div>
          <div style="background:#E8F5E9;border-radius:50%;padding:6px;font-size:20px;">📈</div>
        </div>
        <div style="margin-top:14px;">
          <div class="health-score-num">{score:.1f}<span style="font-size:20px;color:#10B981;font-weight:800;">%</span></div>
          <div style="font-size:12px;color:#64748B;margin-top:-4px;">Overall Health Score</div>
        </div>
        <div class="progress-bar-container">
          <div class="progress-bar-fill" style="width:{bar_w:.0f}%;"></div>
        </div>
        <div class="chip-container">
          <div class="status-chip chip-ph">🧬 pH: {app_state['sensor_ph']:.2f}</div>
          <div class="status-chip chip-humidity">💧 Hum: {app_state['sensor_humidity']}%</div>
          <div class="status-chip chip-temp">🌡️ Temp: {app_state['sensor_temp']}°C</div>
        </div>
        {irr_html}
      </div>
      <div class="mini-grid">
        <div class="mini-card">
          <div class="mini-card-title">Water Level</div>
          <div class="mini-card-val {water_cls}">{water}</div>
        </div>
        <div class="mini-card">
          <div class="mini-card-title">Sunlight</div>
          <div class="mini-card-val {sun_cls}">{sun_text}</div>
          <div style="font-size:10px;color:#94A3B8;margin-top:2px;">{lux_text}</div>
        </div>
        <div class="mini-card">
          <div class="mini-card-title">Soil pH</div>
          <div class="mini-card-val {'val-good' if 6.0<=app_state['sensor_ph']<=7.0 else 'val-warn'}">
            {'✅ Optimal' if 6.0<=app_state['sensor_ph']<=7.0 else '⚠️ Adjust'}
          </div>
        </div>
        <div class="mini-card">
          <div class="mini-card-title">Grove XP</div>
          <div class="mini-card-val val-good">{app_state['xp_score']} pts</div>
        </div>
      </div>
      <div style="margin-top:8px;font-size:10px;color:#94A3B8;text-align:center;">
        KPI targets: Health ≥ 80% | pH 6.0–7.0 | Humidity 40–60% | Temp 20–30°C
      </div>
    </div>"""


def render_diagnosis():
    # 1. Base Image HTML
    if app_state["uploaded_image_b64"]:
        b64 = app_state['uploaded_image_b64']
        img_html = (
            '<div style="position:relative;margin-bottom:12px;">'
            '<img src="data:image/png;base64,' + b64 + '" '
            'style="width:100%;max-height:300px;object-fit:contain;background:#F8FAFC;border-radius:14px;display:block;" '
            'onerror="this.src=\'data:image/jpeg;base64,' + b64 + '\'" />'
            '</div>'
        )
    else:
        img_html   = """
        <div style="width:100%;height:180px;background:linear-gradient(135deg,#D1FAE5,#A7F3D0);
          border-radius:16px;margin-bottom:12px;display:flex;align-items:center;
          justify-content:center;font-size:48px;">🌿</div>"""

    # 2. Merged AI Diagnosis & Dataset Box
    hf_section = ""
    if app_state.get("ai_diagnosis"):
        diag_text = app_state["ai_diagnosis"]

        items = ""
        if app_state.get("hf_results"):
            for r in app_state["hf_results"]:
                # Clean up the text so it doesn't say "Reference match for" again
                clean_label = r["label"].replace("Reference match for: ", "")
                items += (
                    f'<div style="background:#F8FAFC;border-radius:10px;padding:8px 12px;'
                    f'margin-bottom:6px;font-size:12px;color:#334155;border:1px solid #F1F5F9;">'
                    f'📌 Verified Dataset Match: {clean_label}</div>'
                )

        hf_section = f"""
        <div class="main-card" style="margin-top:10px;border-left:4px solid #10B981;">
          <h4 style="margin:0; color:#065F46; font-size:14px;">🤖 AI Diagnosis & Dataset Match</h4>
          <p style="margin:4px 0 10px 0; color:#047857; font-weight:bold; font-size:13px;">{diag_text}</p>
          {items}
          <div style="font-size:10px;color:#94A3B8;margin-top:6px;">
            Dataset: AldoSN/lemon-leaf-disease-dataset (Lemon specific)
          </div>
        </div>
        """

    # 3. Cerebras Agronomist Recommendation Box
    rec_section = ""
    if app_state.get("ai_recommendation"):
        rec_section = f"""
        <div class="main-card" style="margin-top:10px; border-left:4px solid #16A34A;">
          <span style="color:#16A34A;font-weight:bold;font-size:13px;">🧑‍🌾 Agronomist Recommendation</span>
          <p style="font-size:12px;color:#166534;margin:4px 0 0 0;line-height:1.4;">
            {app_state["ai_recommendation"]}
          </p>
        </div>
        """

    # 4. Return Final HTML in the New Order
    return f"""
    <div class="app-container">
      <h2 class="app-title">AI Leaf Diagnosis 🩺</h2>
      <p class="app-subtitle" style="margin-bottom:20px;">Upload a photo · compare with 87k-image HuggingFace dataset</p>

      {img_html}
      {hf_section}
      {rec_section}
    </div>
    """



def render_search_header():
    return """
    <div style="padding:20px 20px 10px 20px;background:#F8FAFC;">
      <h1 class="app-title">Academic Knowledge Base 🔍</h1>
      <p class="app-subtitle">
        RAG-powered search — Firebase inverted index over 5 peer-reviewed citrus papers<br>
        <span style="font-size:11px;color:#94A3B8;">
          Results show matched papers with full abstract excerpts + Cerebras AI summary
        </span>
      </p>
    </div>"""


def render_analytics():
    """
    CHANGES:
    - Top section: KPI status cards (live sensor vs target)
    - Middle section: Interactive time-frame chart (controlled by slider widget)
    - Bottom section: Big Data multi-metric history chart — clearly labelled
    """
    ph  = app_state["sensor_ph"]
    hum = app_state["sensor_humidity"]
    tmp = app_state["sensor_temp"]

    def kpi_card(label, val, lo, hi, unit):
        ok     = lo <= val <= hi
        status = "✅ Optimal" if ok else "⚠️ Out of Range"
        col    = "#10B981"    if ok else "#EF4444"
        fw     = "600"        if ok else "800"
        return (f'<div class="main-card" style="padding:12px;margin-bottom:8px;'
                f'border-left:4px solid {col};">'
                f'<div class="mini-card-title">{label}</div>'
                f'<div style="font-size:16px;font-weight:{fw};color:{col};margin-top:2px;">'
                f'{val}{unit} — {status}</div>'
                f'<div style="font-size:10px;color:#94A3B8;">Target: {lo}–{hi}{unit}</div>'
                f'</div>')

    kpis = (kpi_card("🧬 Soil pH",             ph,  6.0, 7.0, "")   +
            kpi_card("💧 Relative Humidity",    hum, 40,  60,  "%")  +
            kpi_card("🌡️ Ambient Temperature", tmp, 20,  30,  "°C"))

    # Time-frame chart is rendered by the slider widget (see Cell 11)
    # We emit a placeholder here; the widget injects the real image
    n   = app_state.get("chart_n_samples", 7)
    ph_chart  = generate_timeframe_chart_b64("ph")
    hum_chart = generate_timeframe_chart_b64("humidity")
    tmp_chart = generate_timeframe_chart_b64("temp")

    # Big data section
    n_total = len(app_state.get("all_ph", []))
    if n_total >= 3:
        bd_chart = generate_bigdata_chart_b64()
        bd_ph    = app_state["all_ph"];  avg_ph  = sum(bd_ph)  / len(bd_ph)
        bd_hum   = app_state["all_hum"]; avg_hum = sum(bd_hum) / len(bd_hum)
        bd_tmp   = app_state["all_tmp"]; avg_tmp = sum(bd_tmp) / len(bd_tmp)
        n_total = len(app_state.get("all_ph", []))
        n_window = app_state.get("chart_n_samples", 7)

        bd_section = f"""
        <div style="margin-top:24px;">
          <div style="font-size:15px;font-weight:bold;color:#0F172A;margin-bottom:4px;">
            📊 Big Data Analysis — All {n_total} Historical Sensor Readings
          </div>
          <p style="font-size:11px;color:#64748B;margin-bottom:10px;">
            Accumulated across all polling sessions from Firebase.
            Averages: pH {avg_ph:.2f} | Humidity {avg_hum:.1f}% | Temp {avg_tmp:.1f}°C
          </p>
          <div class="main-card" style="text-align:center;padding:10px;">
            <img src="data:image/png;base64,{bd_chart}"
                 style="max-width:100%;border-radius:10px;"/>
          </div>
        </div>"""
    else:
        bd_section = """
        <div style="margin-top:16px;font-size:12px;color:#94A3B8;text-align:center;padding:16px;">
          📊 Big Data chart will appear after you poll sensors at least once.<br>
          Each poll adds readings to the historical dataset.
        </div>"""

    # Generate the PySpark Chart
    pyspark_b64 = generate_pyspark_bigdata_chart_b64()

    html_top = f"""
    <div class="app-container" style="min-height: auto; padding-bottom: 0;">
      <div class="app-header">
        <h1 class="app-title">Data Analytics 📊</h1>
        <p class="app-subtitle">Live KPI cards · Time-frame charts · Full sensor history</p>
      </div>

      <div style="margin-bottom:4px;">
        <div style="font-size:13px;font-weight:bold;color:#0F172A;margin-bottom:8px;">
          🎯 Live Sensor KPIs (current reading vs target range)
        </div>
        {kpis}
      </div>

      <div style="margin-top:20px;margin-bottom:4px;">
        <div style="font-size:13px;font-weight:bold;color:#0F172A;margin-bottom:2px;">
          📈 Time-Frame Charts — last <b>{n}</b> readings
        </div>
        <div style="font-size:11px;color:black;margin-bottom:10px;">
          Use the slider below to change the time window (3 → 30 samples)
        </div>
        <div class="main-card" style="text-align:center;padding:10px;margin-bottom:8px;">
          <img src="data:image/png;base64,{ph_chart}"  style="max-width:100%;border-radius:8px;"/>
        </div>
        <div class="main-card" style="text-align:center;padding:10px;margin-bottom:8px;">
          <img src="data:image/png;base64,{hum_chart}" style="max-width:100%;border-radius:8px;"/>
        </div>
        <div class="main-card" style="text-align:center;padding:10px;margin-bottom:8px;">
          <img src="data:image/png;base64,{tmp_chart}" style="max-width:100%;border-radius:8px;"/>
        </div>
      </div>
    </div>"""

    html_bottom = f"""
    <div class="app-container" style="min-height: auto; padding-top: 10px;">
      {bd_section}

      <div style="margin-top:24px;">
        <div style="font-size:15px;font-weight:bold;color:#0F172A;margin-bottom:4px;">
          🚀 Apache Spark Big Data — Map-Reduce Analysis
        </div>
        <div class="main-card" style="text-align:center;padding:10px;">
          <img src="data:image/png;base64,{pyspark_b64}" style="max-width:100%;border-radius:10px;"/>
        </div>
      </div>
    </div>"""

    return html_top, html_bottom

def render_goals():
    info = get_level_info(app_state["xp_score"])
    c = ["☑️" if app_state["task_completed"][i] else "🔲" for i in range(3)]

    dyn_tasks = ""
    if app_state.get("dynamic_tasks"):
        for t in app_state["dynamic_tasks"]:
            dyn_tasks += f"""
            <div style="display:flex;justify-content:space-between;margin-top:10px;font-size:13px;color:#334155;">
              <span>🔲 <span>{t}</span></span>
              <span style="color:#22C55E;font-weight:bold;">+75 XP</span>
            </div>"""

    done_html = ""
    if app_state.get("done_tasks_log"):
        for d in app_state["done_tasks_log"][:10]:
            done_html += f"""
            <div style="display:flex;justify-content:space-between;padding:6px 0;font-size:13px;
                        color:#334155;border-bottom:1px solid #F1F5F9;">
              <span>✅ {d['name']}</span>
              <span style="color:#94A3B8;">+{d['xp']} XP · {d['time']}</span>
            </div>"""
    else:
        done_html = ('<div style="color:#94A3B8;font-size:13px;text-align:center;padding:16px;">'
                     'No tasks completed yet — go earn some XP! 🌱</div>')

    level_line = ("Max Level Reached! 🎉" if info["is_max"]
                  else f'{info["xp_per_level"] - info["into_level"]} XP to next level')

    return f"""
    <div class="app-container">
      <div class="app-header">
        <h1 class="app-title">Progress & Goals 🏆</h1>
        <p class="app-subtitle">Earn XP for every grove action</p>
      </div>
      <div class="main-card">
        <div style="display:flex;align-items:center;gap:12px;">
          <div style="font-size:24px;">🏆</div>
          <div style="flex-grow:1;">
            <span style="font-weight:bold;color:#0F172A;font-size:14px;">Level {info['level']} — {info['name']}</span><br>
            <span style="font-size:11px;color:#64748B;">{level_line}</span>
          </div>
          <div style="font-size:12px;font-weight:bold;color:#64748B;">{app_state['xp_score']} XP</div>
        </div>
        <div class="progress-bar-container">
          <div class="progress-bar-fill level-fill" style="width:{info['pct']}%;"></div>
        </div>
        <div class="star-rating">{'⭐'*info['level']}<span style="color:#CBD5E1;">{'⭐'*(5-info['level'])}</span></div>
      </div>
      <div style="margin-bottom:16px;">
        <h3 style="font-size:14px;font-weight:bold;color:#0F172A;margin-bottom:8px;">Daily Tasks</h3>
        <div class="main-card" style="padding:12px 16px;">
          <div style="display:flex;justify-content:space-between;margin-bottom:10px;font-size:13px;color:#334155;">
            <span>{c[0]} <span style="{'text-decoration:line-through;color:#94A3B8;' if app_state['task_completed'][0] else ''}">Upload leaf photo & diagnose</span></span>
            <span style="color:#22C55E;font-weight:bold;">+50 XP</span>
          </div>
          <div style="display:flex;justify-content:space-between;margin-bottom:10px;font-size:13px;color:#334155;">
            <span>{c[1]} <span style="{'text-decoration:line-through;color:#94A3B8;' if app_state['task_completed'][1] else ''}">Poll IoT sensors</span></span>
            <span style="color:#22C55E;font-weight:bold;">+50 XP</span>
          </div>
          <div style="display:flex;justify-content:space-between;font-size:13px;color:#334155;">
            <span>{c[2]} <span style="{'text-decoration:line-through;color:#94A3B8;' if app_state['task_completed'][2] else ''}">Run academic search</span></span>
            <span style="color:#22C55E;font-weight:bold;">+50 XP</span>
          </div>
          {dyn_tasks}
        </div>
      </div>
      <div>
        <h3 style="font-size:14px;font-weight:bold;color:#0F172A;margin-bottom:8px;">Done Tasks ✅</h3>
        <div class="main-card" style="padding:10px;">
          {done_html}
        </div>
      </div>
    </div>"""


def render_chat():
    bubbles = ""
    for m in app_state["chat_history"]:
        cls   = "chat-user" if m["role"] == "user" else "chat-bot"
        emoji = "🧑‍🌾"        if m["role"] == "user" else "🤖"
        bubbles += f'<div class="{cls}">{emoji} {m["content"]}</div>'
    if not bubbles:
        bubbles = ('<div style="color:#94A3B8;font-size:13px;text-align:center;padding:20px;">'
                   'Ask me anything about your lemon tree! 🌿</div>')
    return f"""
    <div class="app-container">
      <div class="app-header">
        <h1 class="app-title">Lemon Pulse AI 🤖</h1>
        <p class="app-subtitle">Powered by Cerebras — expert agronomist chatbot</p>
      </div>
      <div class="main-card" style="padding:0;overflow:hidden;">
        <div class="chat-wrap">{bubbles}</div>
      </div>
    </div>"""


In [ ]:
#@title Interactive Widgets & Action Handlers
sample_slider = widgets.IntSlider(
    value=7, min=3, max=30, step=1,
    description="Samples:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="100%", margin="4px 0"),
)

#  IoT Poll button
btn_poll = widgets.Button(
    description="🔄 Poll IoT Sensors",
    button_style="info",
    layout=widgets.Layout(width="100%", margin="8px 0"),
)

def on_poll_sensors(b):
    n = sample_slider.value
    app_state["sample_count"] = n

    # Show loading message directly in output_view (no nested Output widget)
    with output_view:
        clear_output(wait=True)
        display(HTML(
            f"<div style='padding:24px 20px;color:#2563EB;font-weight:600;font-size:14px;"
            f"background:#EFF6FF;border-radius:12px;margin:16px;'>"
            f"🔄 Fetching {n} sensor samples from Firebase…<br>"
            f"<span style='font-size:11px;color:#64748B;'>This may take up to 30s if server is sleeping.</span>"
            f"</div>"
        ))

    try:
        feeds = {}
        for feed in ["temperature", "humidity", "soil"]:
            r = requests.get(FIREBASE_SENSOR_URL,
                             params={"feed": feed, "limit": n}, timeout=35)
            feeds[feed] = [float(i["value"]) for i in r.json().get("data", [])]

        raw_t = feeds.get("temperature", []) or [round(24.0 + random.uniform(-1.0, 1.0), 1) for _ in range(n)]
        raw_h = feeds.get("humidity",    []) or [round(45.0 + random.uniform(-3.0, 3.0), 1) for _ in range(n)]
        raw_s = feeds.get("soil",        []) or [round(65.0 + random.uniform(-5.0, 5.0), 1) for _ in range(n)]

        while len(raw_t) < n: raw_t.append(24.0)
        while len(raw_h) < n: raw_h.append(45.0)
        while len(raw_s) < n: raw_s.append(65.0)

        for lst in [raw_t, raw_h, raw_s]: lst.reverse()

        ph_vals = [max(5.5, min(7.5, 6.0 + (v % 1.0 if v > 14 else v / 100.0)))
                   for v in raw_s]

        # Update live readings
        app_state["sensor_temp"]      = int(raw_t[-1])
        app_state["sensor_humidity"]  = int(raw_h[-1])
        app_state["sensor_ph"]        = float(ph_vals[-1])
        app_state["history_temp"]     = [int(x) for x in raw_t[-7:]]
        app_state["history_humidity"] = [int(x) for x in raw_h[-7:]]
        app_state["history_ph"]       = [float(x) for x in ph_vals[-7:]]

        # Accumulate for Big Data
        app_state["all_ph"]  += ph_vals
        app_state["all_hum"] += [int(x) for x in raw_h]
        app_state["all_tmp"] += [int(x) for x in raw_t]

        _save_history_to_firebase()

        complete_daily_task(1, "Poll IoT sensors")

        # Print samples to cell output (not UI) for debugging
        print(f"\n✅ Poll complete — {n} samples received")
        print(f"   pH:       {[round(x,2) for x in ph_vals]}")
        print(f"   Humidity: {[int(x) for x in raw_h]}")
        print(f"   Temp:     {[int(x) for x in raw_t]}")
        print(f"   Latest → pH={app_state['sensor_ph']:.2f}  "
              f"Hum={app_state['sensor_humidity']}%  "
              f"Temp={app_state['sensor_temp']}°C")

        update_display()
    except Exception as e:
        print(f"❌ Poll failed: {e}")
        update_display()

    # Weather sync
    weather = fetch_weather_forecast()
    app_state["weather_cache"] = weather
    if weather:
        status, msg = evaluate_irrigation(weather, app_state["sensor_humidity"])
        app_state["irrigation_status"] = (status, msg)
        print(f"   Weather: {weather['description']}")
        print(f"   Irrigation: {status.upper()}")

    update_display()

btn_poll.on_click(on_poll_sensors)


def _save_history_to_firebase():
    """Save accumulated sensor history to Firebase so Big Data persists across Colab sessions."""
    try:
        payload = {
            "ph" : app_state["all_ph"],
            "hum": app_state["all_hum"],
            "tmp": app_state["all_tmp"],
        }
        requests.put(FIREBASE_HISTORY_URL, json=payload, timeout=8)
    except Exception as e:
        print(f"[Firebase save] {e}")


def _load_history_from_firebase():
    """Load accumulated sensor history from Firebase on app startup."""
    try:
        r = requests.get(FIREBASE_HISTORY_URL, timeout=8)
        if r.status_code == 200 and r.json():
            d = r.json()
            app_state["all_ph"]  = d.get("ph",  [])
            app_state["all_hum"] = d.get("hum", [])
            app_state["all_tmp"] = d.get("tmp", [])
            print(f"✅ Loaded {len(app_state['all_ph'])} historical readings from Firebase.")
    except Exception as e:
        print(f"[Firebase load] {e}")

_load_history_from_firebase()


#  Leaf Upload + Clear
upload_widget = widgets.FileUpload(
    accept="image/*", multiple=False,
    description="📷 Upload Leaf Photo",
    layout=widgets.Layout(width="100%", margin="8px 0"),
)

btn_clear_image = widgets.Button(
    description="✕ Clear Image",
    button_style="danger",
    layout=widgets.Layout(width="100%", margin="4px 0"),
)

btn_diagnose = widgets.Button(
    description="✨ Run AI Diagnosis",
    button_style="warning",
    layout=widgets.Layout(width="60%", margin="0 auto", display="block"),
)

btn_close_diag = widgets.Button(
    description="💬 Discuss with Chatbot",
    button_style="info",
    layout=widgets.Layout(width="100%", margin="4px 0"),

)


btn_add_dynamic_goal = widgets.Button(
    description="➕ Add Action to Goals",
    button_style="success",
    layout=widgets.Layout(width="60%", margin="0 auto", display="block"),
)

def on_add_dynamic_goal(b):
    if "dynamic_tasks" not in app_state:
        app_state["dynamic_tasks"] = []

    task = app_state.get("ai_task")
    if task and task not in app_state["dynamic_tasks"]:
        app_state["dynamic_tasks"].append(task)
        app_state["ai_task"] = None # Hide the button once added
        trigger_tab("Diagnosis")

btn_add_dynamic_goal.on_click(on_add_dynamic_goal)



def on_file_upload(change):
    uploaded = upload_widget.value
    if not uploaded:
        return
    # Handle both old dict format and new list/tuple format
    if isinstance(uploaded, dict):
        fi = list(uploaded.values())[0]
        content = fi['content']
    elif isinstance(uploaded, (list, tuple)) and len(uploaded) > 0:
        fi = uploaded[0]
        content = fi.get('content', b'')
    else:
        return
    if not content:
        return
    app_state["uploaded_image_b64"] = base64.b64encode(bytes(content)).decode()
    complete_daily_task(0, "Uploading file")
    app_state["hf_results"]         = []
    trigger_tab("Diagnosis")

upload_widget.observe(on_file_upload, names='value')

def on_clear_image(b):
   # Wipe the photo
    app_state["uploaded_image_b64"] = None
    app_state["hf_results"] = []
    app_state["ai_diagnosis"] = None
    app_state["ai_recommendation"] = None
    app_state["ai_task"] = None
    app_state["upload_prompt"] = "Upload a leaf photo to begin"

    # reset the read-only FileUpload widget via set_trait
    upload_widget.set_trait('value', {})
    upload_widget._counter = 0

    trigger_tab("Diagnosis")



def on_hf_search(b):
    # Check if an image is actually loaded in the app state
    if not app_state.get("uploaded_image_b64"):
        app_state["upload_prompt"] = "⚠️ Please upload an image before running AI Diagnosis!"
        trigger_tab("Diagnosis")
        return

    with output_view:
        clear_output(wait=True)
        print("1. Button clicked. Extracting image...")


        # Safely extract image bytes
        try:
            if isinstance(upload_widget.value, dict):
                uploaded_file = list(upload_widget.value.values())[0]
            else:
                uploaded_file = upload_widget.value[0]
            image_bytes = uploaded_file['content']
            print("2. Image extracted. Running local Hugging Face AI...")
        except Exception as e:
            print(f"Error extracting image: {e}")
            return

        # Run Local Inference
        predictions = analyze_image_locally(image_bytes)

        if predictions and len(predictions) > 0:
            raw_label = predictions[0].get('label', 'Unknown')
            confidence = predictions[0].get('score', 0) * 100

            clean_label = str(raw_label).replace("___", " - ").replace("_", " ")
            app_state["ai_diagnosis"] = f"{clean_label} ({confidence:.1f}% Confidence)"
            query = clean_label.split("-")[0].strip()

            print(f"3. Identified: {query}. Consulting Cerebras Agronomist AI...")

            # CEREBRAS INTEGRATION
            system_prompt = (
                "You are an expert citrus agronomist AI. Given a diagnosed lemon disease, "
                "provide exactly two things:\\n"
                "1. A 2-sentence treatment recommendation.\\n"
                "2. A specific, actionable 5-10 word task for the farmer starting with 'TASK:'\\n"
                "Do not include any other formatting."
            )
            user_msg = f"My lemon leaf was diagnosed with {query}."

            try:
                ai_resp = call_cerebras(system_prompt, user_msg)
                if ai_resp and "LLM Error" in ai_resp:
                    app_state["ai_recommendation"] = "The AI is currently processing high loads. Please try again or apply a general citrus fungicide."
                    app_state["ai_task"] = "Apply general fungicide"
                elif ai_resp and "TASK:" in ai_resp:
                    rec, task = ai_resp.split("TASK:", 1)
                    app_state["ai_recommendation"] = rec.strip()
                    app_state["ai_task"] = task.strip()
                else:
                    app_state["ai_recommendation"] = ai_resp.strip() if ai_resp else "No recommendation generated."
                    app_state["ai_task"] = None
            except Exception as e:
                app_state["ai_recommendation"] = "Cerebras AI could not be reached."
                app_state["ai_task"] = None

        else:
            app_state["ai_diagnosis"] = "Local AI failed to process image."
            query = "healthy"

        print("4. Fetching reference images...")
        results = search_hf_plant_disease(query_label=query, max_results=1)
        app_state["hf_results"] = results if results else [
            {"label": f"Fallback search: {query}", "image": None}
        ]

        print("5. Triggering UI update...")
        trigger_tab("Diagnosis")

def on_close_diag(b):
    """Close diagnosis: wipe image+results, then offer chatbot follow-up."""

    if not app_state.get("uploaded_image_b64"):
        return

    if app_state["uploaded_image_b64"]:
        label = classify_leaf_locally(app_state["uploaded_image_b64"])
    else:
        label = None

    # Wipe everything
    app_state["uploaded_image_b64"] = None
    app_state["hf_results"] = []
    app_state["ai_diagnosis"] = None
    app_state["ai_recommendation"] = None
    app_state["ai_task"] = None
    app_state["upload_prompt"] = "Upload a leaf photo to begin"

    if label and label != "Upload a leaf photo to begin":
        # Auto-inject diagnosis into chatbot as opening message
        auto_msg = (f"I just diagnosed my lemon tree leaf. "
                    f"The result was: '{label}'. "
                    f"What should I do next to treat this condition?")
        app_state["chat_history"].append({"role": "user", "content": auto_msg})
        # Get AI response
        reply = chatbot_reply(auto_msg)
        app_state["chat_history"].append({"role": "assistant", "content": reply})
        # Go to chatbot with the pre-loaded conversation
        trigger_tab("Chat")
    else:
        trigger_tab("Home")
btn_add_task = widgets.Button(
    description="🌟 Claim 50 XP (Daily Task)",
    button_style="success",
    layout=widgets.Layout(width="100%", margin="4px 0"),
)

def on_add_task(b):
    if not app_state.get("uploaded_image_b64"):
        return

    complete_daily_task(0, "Adding Task ")
    trigger_tab("Goals")
btn_mark_task0 = widgets.Button(description="✅ Mark 'Upload photo' Done", button_style="success", layout=widgets.Layout(width="100%", margin="2px 0"))
btn_mark_task1 = widgets.Button(description="✅ Mark 'Poll sensors' Done", button_style="success", layout=widgets.Layout(width="100%", margin="2px 0"))
btn_mark_task2 = widgets.Button(description="✅ Mark 'Academic search' Done", button_style="success", layout=widgets.Layout(width="100%", margin="2px 0"))
btn_complete_dynamic = widgets.Button(description="✅ Mark Next AI Task Done", button_style="success", layout=widgets.Layout(width="100%", margin="8px 0 2px 0"))

def on_mark_task0(b): complete_daily_task(0, "Upload leaf photo & diagnose"); trigger_tab("Goals")
def on_mark_task1(b): complete_daily_task(1, "Poll IoT sensors"); trigger_tab("Goals")
def on_mark_task2(b): complete_daily_task(2, "Run academic search"); trigger_tab("Goals")
def on_complete_dynamic(b):
    tasks = app_state.get("dynamic_tasks", [])
    if tasks:
        complete_dynamic_task(tasks[0])
    trigger_tab("Goals")

btn_mark_task0.on_click(on_mark_task0)
btn_mark_task1.on_click(on_mark_task1)
btn_mark_task2.on_click(on_mark_task2)
btn_complete_dynamic.on_click(on_complete_dynamic)
btn_add_task.on_click(on_add_task)
btn_clear_image.on_click(on_clear_image)
btn_diagnose.on_click(on_hf_search)
btn_close_diag.on_click(on_close_diag)


#  RAG Search
search_input = widgets.Text(
    value="citrus leafminer treatment",
    placeholder="e.g. pH, leafminer, irrigation, chlorosis…",
    layout=widgets.Layout(width="100%", margin="0 0 8px 0"),
)
btn_search = widgets.Button(
    description="🔍 Search + Generate AI Advice",
    button_style="primary",
    layout=widgets.Layout(width="100%", height="40px"),
)
rag_output = widgets.Output()

def on_search(b):
    with rag_output:
        clear_output()
        query = search_input.value.strip()
        if not query:
            display(HTML("<div style='color:#EF4444;padding:8px;'>⚠️ Enter a query.</div>"))
            return

        display(HTML(f"<div style='color:#2563EB;padding:8px;font-weight:600;'>"
                     f"🔄 Searching Firebase index for: '{query}'…</div>"))
        results = search_documents(query)
        clear_output()

        is_fallback = not results
        if not is_fallback:
            best       = results[0]
            rag_prompt = (
                f"User asked: \"{query}\"\n\n"
                f"Reference: {best['title']}\nDomain: {best['domain']}\n"
                f"Summary: {best['summary']}\n\n"
                "Give a 2–3 sentence agronomic answer with 2–3 bullet-point action items. "
                "Be practical for a lemon farmer."
            )
        else:
            ctx        = "\n".join(f"Doc {k}: {v['title']} — {v['summary']}"
                                   for k, v in DOCUMENTS.items())
            rag_prompt = (
                f"User asked: \"{query}\"\nNo direct index match. "
                f"Select the most relevant from:\n{ctx}\n\n"
                "2–3 sentence answer + 2–3 bullet actions. End with [SELECTED_DOC_ID: X]"
            )

        ai_text = call_cerebras(
            "You are an expert AI agronomist for Lemon Pulse.", rag_prompt, 0.2)

        selected_id = results[0]["id"] if not is_fallback else 1
        if is_fallback:
            m = re.search(r'\[SELECTED_DOC_ID:\s*(\d+)\]', ai_text)
            if m: selected_id = int(m.group(1))
            ai_text = re.sub(r'\[SELECTED_DOC_ID:\s*\d+\]', '', ai_text).strip()

        doc_meta  = DOCUMENTS.get(selected_id, DOCUMENTS[1])
        badge_col = "#10B981" if not is_fallback else "#F59E0B"
        badge_bg  = "#F0FDF4" if not is_fallback else "#FFFBEB"
        src_lbl   = "Primary Index Match" if not is_fallback else "AI Semantic Fallback"

        #  AI answer card
        display(HTML(f"""
        <div class="main-card"
             style="background:{badge_bg};border-left:5px solid {badge_col};padding:16px;margin-bottom:12px;">
          <div style="font-size:11px;font-weight:800;color:{badge_col};
                      text-transform:uppercase;margin-bottom:6px;">
            🧠 {src_lbl}
          </div>
          <p style="font-size:13px;color:#14532D;margin:0;line-height:1.6;">{ai_text}</p>
        </div>"""))

        #  All papers ranked with abstract
        results_map = {r["id"]: r for r in results}
        all_docs    = sorted(DOCUMENTS.items(),
                             key=lambda x: results_map.get(x[0], {}).get("score", 0),
                             reverse=True)

        display(HTML('<div style="font-size:12px;font-weight:bold;color:#64748B;'
                     'margin:16px 0 8px;">📊 All Publications — Ranked by Index Score:</div>'))

        for doc_id, meta in all_docs:
            r         = results_map.get(doc_id)
            is_best   = (doc_id == selected_id)
            border    = (f"border-left:5px solid {badge_col};" if is_best else
                         "border-left:4px solid #10B981;"      if r else
                         "border-left:4px solid #CBD5E1;opacity:0.7;")
            prefix    = ("⭐ [PRIMARY] " if is_best else
                         "✅ [MATCHED] "  if r else
                         "📄 [NO MATCH] ")
            score_str = f"Score: {r['score']} | Terms: {', '.join(r['term_frequencies'].keys())}" if r else "No index match"
            rank_str  = f"Tutorial rank: {r['tutorial_rank']*100:.1f}%" if r else ""

            # Show abstract excerpt (first 220 chars)
            abstract_excerpt = meta.get("abstract", "")[:220].rstrip() + "…"

            display(HTML(f"""
            <div class="main-card"
                 style="padding:14px;margin-bottom:10px;{border}">
              <a href="{meta['url']}" target="_blank"
                 style="color:#1E40AF;font-size:13px;font-weight:bold;text-decoration:none;">
                {prefix}{meta['title']}
              </a>
              <div style="font-size:11px;color:#64748B;margin-top:3px;">
                Domain: <i>{meta['domain']}</i>
              </div>
              <div style="font-size:11.5px;color:#334155;margin-top:8px;
                          background:#F8FAFC;border-radius:8px;padding:8px;
                          border:1px solid #F1F5F9;line-height:1.5;">
                {abstract_excerpt}
              </div>
              <div style="font-size:11px;color:#94A3B8;margin-top:6px;">
                {score_str}  {rank_str}
              </div>
            </div>"""))

        complete_daily_task(2, "Searching ")

btn_search.on_click(on_search)

search_panel = widgets.VBox(
    [search_input, btn_search, rag_output],
    layout=widgets.Layout(padding="0 20px 20px 20px"),
)
search_panel.add_class("lemon-app")


#  Analytics time-frame slider
timeframe_buttons = widgets.ToggleButtons(
    options=[("Last Hour", "1h"), ("Last 6h", "6h"),
             ("Last Day", "24h"), ("Last Week", "7d"), ("All Data", "all")],
    value="24h",
    description="",
    style={"button_width": "80px", "description_width": "0px"},
    layout=widgets.Layout(width="100%", margin="8px 0"),
)

def _samples_for_timeframe(tf):
    """
    Map time-frame label to a reading count.
    Assumes sensor polls happen roughly every 5 minutes from ESP32.
    Adjust READINGS_PER_HOUR if your polling interval differs.
    """
    READINGS_PER_HOUR = 12   # 1 reading per 5 min → 12/hour
    mapping = {
        "1h" : READINGS_PER_HOUR,
        "6h" : READINGS_PER_HOUR * 6,
        "24h": READINGS_PER_HOUR * 24,
        "7d" : READINGS_PER_HOUR * 24 * 7,
        "all": 99999,
    }
    return mapping.get(tf, 24)

def on_timeframe_change(change):
    app_state["chart_timeframe"] = change["new"]
    if app_state["current_tab"] == "Analytics":
        update_display()

timeframe_buttons.observe(on_timeframe_change, names="value")
app_state["chart_timeframe"] = "24h"

#  Chatbot
chat_input    = widgets.Text(
    placeholder="Ask about your lemon tree…",
    layout=widgets.Layout(width="79%"),
)
btn_send_chat = widgets.Button(
    description="Send 📨",
    button_style="success",
    layout=widgets.Layout(width="20%", margin="0 0 0 1%"),
)
btn_clear_chat = widgets.Button(
    description="Clear Chat 🗑️",
    layout=widgets.Layout(width="100%", margin="4px 0"),
)
chat_controls = widgets.HBox(
    [chat_input, btn_send_chat],
    layout=widgets.Layout(width="100%", padding="0 20px 8px 20px"),
)
chat_controls.add_class("lemon-app")

def on_send_chat(b):
    msg = chat_input.value.strip()
    if not msg:
        return
    chat_input.value = ""
    app_state["chat_history"].append({"role": "user", "content": msg})
    with output_view:
        clear_output(wait=True)
        display(HTML(render_chat()))
        display(HTML("<div style='padding:8px 20px;color:#7C3AED;font-size:12px;'>🤖 Thinking…</div>"))
    reply = chatbot_reply(msg)
    app_state["chat_history"].append({"role": "assistant", "content": reply})
    trigger_tab("Chat")

def on_clear_chat(b):
    app_state["chat_history"] = []
    trigger_tab("Chat")

btn_send_chat.on_click(on_send_chat)
btn_clear_chat.on_click(on_clear_chat)



In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

#@title  Navigation Engine & App Assembly
def update_display():
    """Render the active tab. clear_output(wait=True) prevents ghost output."""
    with output_view:
        clear_output(wait=True)
        tab = app_state["current_tab"]
        if tab == "Home":
            display(HTML(render_home()))
            display(widgets.VBox([sample_slider, btn_poll]))
        elif tab == "Diagnosis":
            display(HTML(render_diagnosis()))

            elements = []
            if app_state.get("ai_task"):
                elements.append(widgets.HBox(
                    [btn_add_dynamic_goal],
                    layout=widgets.Layout(width="100%", justify_content="center", padding="4px 16px")
                ))

            diag_box = widgets.HBox(
                      [btn_diagnose],
                      layout=widgets.Layout(width="100%", justify_content="center", padding="12px 16px 8px 16px")
                  )

            row1 = widgets.HBox(
                      [upload_widget, btn_clear_image],
                      layout=widgets.Layout(width="100%", padding="0 16px", gap="6px")
                  )

            if app_state.get("ai_diagnosis"):
                row2 = widgets.HBox(
                    [btn_add_task, btn_close_diag],
                    layout=widgets.Layout(width="100%", padding="8px 16px 12px 16px", gap="6px")
                )
                display(widgets.VBox([diag_box] + elements + [row1, row2], layout=widgets.Layout(background_color="#FFFFFF")))
            else:
                # If no diagnosis, just show the upload and clear buttons
                display(widgets.VBox([diag_box] + elements + [row1], layout=widgets.Layout(background_color="#FFFFFF")))


        elif tab == "Search":
            display(HTML(render_search_header()))
            display(search_panel)
            on_search(None)
        elif tab == "Goals":
              display(HTML(render_goals()))
              task_buttons = []
              if not app_state["task_completed"][0]: task_buttons.append(btn_mark_task0)
              if not app_state["task_completed"][1]: task_buttons.append(btn_mark_task1)
              if not app_state["task_completed"][2]: task_buttons.append(btn_mark_task2)
              if app_state.get("dynamic_tasks"):     task_buttons.append(btn_complete_dynamic)
        elif tab == "Analytics":
            top_html, bottom_html = render_analytics()
            display(HTML(top_html))
            display(timeframe_buttons)
            display(HTML(bottom_html))
        elif tab == "Chat":
            display(HTML(render_chat()))
            display(chat_controls)
            display(widgets.VBox([btn_clear_chat],
                                  layout=widgets.Layout(padding="0 20px")))


def trigger_tab(name):
    app_state["current_tab"] = name
    active = {n: "" for n in ["Home","Diagnosis","Search","Goals","Analytics","Chat"]}
    active[name] = "success"
    for btn, key in [(nav_home,"Home"),(nav_diag,"Diagnosis"),(nav_srch,"Search"),
                     (nav_goal,"Goals"),(nav_anal,"Analytics"),(nav_chat,"Chat")]:
        btn.button_style = active[key]
    update_display()


# Navigation bar
_L = widgets.Layout(width="auto", flex="1")
nav_home = widgets.Button(description="🏠 Home",      layout=_L)
nav_diag = widgets.Button(description="🩺 Diagnose",  layout=_L)
nav_srch = widgets.Button(description="🔍 Search",    layout=_L)
nav_goal = widgets.Button(description="🏆 Goals",     layout=_L)
nav_anal = widgets.Button(description="📊 Analytics", layout=_L)
nav_chat = widgets.Button(description="🤖 AI Chat",   layout=_L)

for btn in [nav_home, nav_diag, nav_srch, nav_goal, nav_anal, nav_chat]:
    btn.add_class("nav-btn")

nav_home.on_click(lambda b: trigger_tab("Home"))
nav_diag.on_click(lambda b: trigger_tab("Diagnosis"))
nav_srch.on_click(lambda b: trigger_tab("Search"))
nav_goal.on_click(lambda b: trigger_tab("Goals"))
nav_anal.on_click(lambda b: trigger_tab("Analytics"))
nav_chat.on_click(lambda b: trigger_tab("Chat"))
nav_home.button_style = "success"

nav_bar = widgets.HBox(
    [nav_home, nav_diag, nav_srch, nav_goal, nav_anal, nav_chat],
    layout=widgets.Layout(justify_content="space-around",
                          padding="10px 0", background_color="white"),
)
app_frame = widgets.VBox(
    [nav_bar, output_view],
    layout=widgets.Layout(max_width="900px", margin="0 auto",
                          border="1px solid #CBD5E1", border_radius="12px",
                          overflow="hidden", background_color="#F8FAFC"),
)
app_frame.add_class("lemon-app")

#  Launch
display(HTML(CSS_STYLE))   # inject fresh CSS on every run
display(app_frame)
update_display()